# 01 — Data Collection
This notebook downloads 10-K annual filings from SEC EDGAR for a sample of S&P 500 companies
between 2015 and 2023. Filings are saved to `data/raw/` for downstream processing.



In [1]:
import os
import time
import pandas as pd
from sec_edgar_downloader import Downloader

# Create directories
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/cleaned", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

print("Directories ready.")

Directories ready.


In [4]:
# Top 50 S&P 500 companies by market cap (as of 2024)
companies = [
    # Technology
    ("AAPL", "Technology"), ("MSFT", "Technology"), ("NVDA", "Technology"),
    ("GOOGL", "Technology"), ("META", "Technology"), ("AVGO", "Technology"),
    ("ORCL", "Technology"), ("CRM", "Technology"), ("AMD", "Technology"),
    ("ADBE", "Technology"),

    # Consumer Discretionary
    ("AMZN", "Consumer Discretionary"), ("TSLA", "Consumer Discretionary"),
    ("HD", "Consumer Discretionary"), ("MCD", "Consumer Discretionary"),
    ("NKE", "Consumer Discretionary"),

    # Financials
    ("BRK-B", "Financials"), ("JPM", "Financials"), ("V", "Financials"),
    ("MA", "Financials"), ("BAC", "Financials"), ("WFC", "Financials"),
    ("GS", "Financials"),

    # Healthcare
    ("LLY", "Healthcare"), ("UNH", "Healthcare"), ("JNJ", "Healthcare"),
    ("ABBV", "Healthcare"), ("MRK", "Healthcare"), ("TMO", "Healthcare"),
    ("ABT", "Healthcare"),

    # Energy
    ("XOM", "Energy"), ("CVX", "Energy"),

    # Industrials
    ("CAT", "Industrials"), ("RTX", "Industrials"), ("HON", "Industrials"),
    ("GE", "Industrials"), ("UPS", "Industrials"),

    # Communication Services
    ("NFLX", "Communication Services"), ("DIS", "Communication Services"),
    ("T", "Communication Services"), ("VZ", "Communication Services"),

    # Consumer Staples
    ("WMT", "Consumer Staples"), ("PG", "Consumer Staples"),
    ("KO", "Consumer Staples"), ("PEP", "Consumer Staples"),
    ("COST", "Consumer Staples"),

    # Utilities
    ("NEE", "Utilities"), ("DUK", "Utilities"),

    # Materials
    ("LIN", "Materials"), ("APD", "Materials"),
]

df_companies = pd.DataFrame(companies, columns=["ticker", "sector"])
print(f"Company universe: {len(df_companies)} companies across {df_companies['sector'].nunique()} sectors")
print()
print(df_companies.groupby("sector").size().sort_values(ascending=False).to_string())

Company universe: 49 companies across 10 sectors

sector
Technology                10
Healthcare                 7
Financials                 7
Consumer Discretionary     5
Industrials                5
Consumer Staples           5
Communication Services     4
Energy                     2
Materials                  2
Utilities                  2


In [5]:
# Initialize downloader — SEC requires a name and email for the user-agent header
dl = Downloader("Ted Shabecoff", "ted@looplens.co", "../data/raw")

# Download 10-K filings for each company (2015–2023)
failed = []

for _, row in df_companies.iterrows():
    ticker = row["ticker"]
    try:
        dl.get("10-K", ticker, after="2015-01-01", before="2023-12-31")
        print(f"✓ {ticker}")
        time.sleep(0.5)  # Be polite to SEC servers
    except Exception as e:
        print(f"✗ {ticker}: {e}")
        failed.append(ticker)

print(f"\nDone. Failed tickers: {failed if failed else 'None'}")

✓ AAPL
✓ MSFT
✓ NVDA
✓ GOOGL
✓ META
✓ AVGO
✓ ORCL
✓ CRM
✓ AMD
✓ ADBE
✓ AMZN
✓ TSLA
✓ HD
✓ MCD
✓ NKE
✓ BRK-B
✓ JPM
✓ V
✓ MA
✓ BAC
✓ WFC
✓ GS
✓ LLY
✓ UNH
✓ JNJ
✓ ABBV
✓ MRK
✓ TMO
✓ ABT
✓ XOM
✓ CVX
✓ CAT
✓ RTX
✓ HON
✓ GE
✓ UPS
✓ NFLX
✓ DIS
✓ T
✓ VZ
✓ WMT
✓ PG
✓ KO
✓ PEP
✓ COST
✓ NEE
✓ DUK
✓ LIN
✓ APD

Done. Failed tickers: None


In [6]:
# Check what was actually downloaded
downloaded = []

for ticker in df_companies["ticker"]:
    ticker_path = f"../data/raw/sec-edgar-filings/{ticker}/10-K"
    if os.path.exists(ticker_path):
        filings = os.listdir(ticker_path)
        downloaded.append({"ticker": ticker, "filings_count": len(filings)})
    else:
        downloaded.append({"ticker": ticker, "filings_count": 0})

df_downloaded = pd.DataFrame(downloaded)
print(df_downloaded.to_string(index=False))
print(f"\nTotal filings downloaded: {df_downloaded['filings_count'].sum()}")

ticker  filings_count
  AAPL              9
  MSFT              9
  NVDA              9
 GOOGL              8
  META              9
  AVGO              6
  ORCL              9
   CRM              9
   AMD              9
  ADBE              9
  AMZN              9
  TSLA              9
    HD              9
   MCD              9
   NKE              9
 BRK-B              9
   JPM              9
     V              9
    MA              9
   BAC              9
   WFC              9
    GS              9
   LLY              9
   UNH              9
   JNJ              9
  ABBV              9
   MRK              9
   TMO              9
   ABT              9
   XOM              9
   CVX              9
   CAT              9
   RTX              9
   HON              9
    GE              9
   UPS              9
  NFLX              9
   DIS              5
     T              9
    VZ              9
   WMT              9
    PG              9
    KO              9
   PEP              9
  COST    